# Paired *t*-tests: baselines vs frozen CLIP image-only (one-sided)

30 matched seeds; **d = method − frozen CLIP** on test macro-F1 and test accuracy.

Comparisons (Section 3c of progress report):
- Attention fusion + LLaVA (`phase3_robustness`)
- ViT-B/16 image-only (`imageonly_vit_b16_robustness`)
- Partial fine-tuned CLIP, last 2 blocks (`imageonly_clip_finetuned_robustness`)

In [1]:
import json
import os
import re
import statistics
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(12):
    if (_p / "experiments").is_dir() and (_p / "data").is_dir():
        PROJECT_ROOT = _p
        break
    _p = _p.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

from scipy.stats import ttest_1samp

BASELINE_EXP = "imageonly_robustness"
ALPHA = 0.05

METHODS = [
    ("phase3_robustness", "Attention fusion + LLaVA"),
    ("imageonly_vit_b16_robustness", "ViT-B/16 image-only"),
    ("imageonly_clip_finetuned_robustness", "Partial fine-tuned CLIP (2 blocks)"),
]

METRICS = [
    ("test_macro_f1", "Test Macro F1", False),
    ("test_accuracy", "Test Accuracy", True),
]


def load_test_metric(exp_name: str, json_key: str, as_fraction: bool = False) -> dict[int, float]:
    out: dict[int, float] = {}
    base = PROJECT_ROOT / "experiments" / exp_name / "metrics" / "experiments"
    for path in base.glob("seed_*_results.json"):
        m = re.match(r"seed_(\d+)_results\.json$", path.name)
        if not m:
            continue
        obj = json.loads(path.read_text(encoding="utf-8"))
        v = float(obj["test_metrics"][json_key])
        if as_fraction and v > 1.0:
            v /= 100.0
        out[int(m.group(1))] = v
    return out


for json_key, label, as_fraction in METRICS:
    baseline = load_test_metric(BASELINE_EXP, json_key, as_fraction)
    print("=" * 60)
    print(label)
    print("baseline_mean", statistics.mean(baseline.values()))
    print()

    for exp_name, method_label in METHODS:
        method = load_test_metric(exp_name, json_key, as_fraction)
        seeds = sorted(baseline.keys() & method.keys())
        if baseline.keys() != method.keys():
            raise ValueError(f"{method_label}: seed mismatch with baseline")

        d = [method[s] - baseline[s] for s in seeds]
        mean_d = statistics.mean(d)
        std_d = statistics.stdev(d) if len(d) > 1 else float("nan")
        cohen_d = mean_d / std_d if std_d else float("nan")
        wins = sum(1 for x in d if x > 0)
        ties = sum(1 for x in d if x == 0)
        losses = sum(1 for x in d if x < 0)

        res = ttest_1samp(d, popmean=0.0, alternative="greater")

        print(method_label)
        print("  method_mean", statistics.mean(method.values()))
        print("  mean_diff", mean_d)
        print("  stdev_diff", std_d)
        print("  paired_Cohen_d", cohen_d)
        print("  wins_ties_losses", wins, ties, losses)
        print("  t", res.statistic)
        print("  p_one_sided", res.pvalue)
        print("  reject_H0_method_gt_baseline", res.pvalue < ALPHA)
        print()

Test Macro F1
baseline_mean 0.8145000550248405

Attention fusion + LLaVA
  method_mean 0.8310843219571423
  mean_diff 0.016584266932301864
  stdev_diff 0.010423274665449109
  paired_Cohen_d 1.5910802952622085
  wins_ties_losses 27 0 3
  t 8.714705685170916
  p_one_sided 6.788187343180793e-10
  reject_H0_method_gt_baseline True

ViT-B/16 image-only
  method_mean 0.8338012818611278
  mean_diff 0.019301226836287332
  stdev_diff 0.008552234516401832
  paired_Cohen_d 2.256863606729988
  wins_ties_losses 29 0 1
  t 12.361351066184826
  p_one_sided 2.196019698288449e-13
  reject_H0_method_gt_baseline True

Partial fine-tuned CLIP (2 blocks)
  method_mean 0.8505172291735119
  mean_diff 0.03601717414867153
  stdev_diff 0.008771346401314184
  paired_Cohen_d 4.1062309593969735
  wins_ties_losses 30 0 0
  t 22.49075322787802
  p_one_sided 3.333260083849447e-20
  reject_H0_method_gt_baseline True

Test Accuracy
baseline_mean 0.8152549094359614

Attention fusion + LLaVA
  method_mean 0.8311838790931